# 一次异常读数，会把估计拉偏多少？

**实验手记 01**　·　本地 CPU　·　Python 3 / IPython / Matplotlib

> 连测九次温度，最后一次突然偏高。去掉最高和最低各一个读数，再求平均，结果会怎样？

本节先看数据，再定义并调用方法，最后比较误差。内核就绪后，目标是约 5 秒内得到一次小观察。使用 **Restart Kernel and Run All Cells** 从头复跑，或 **Shift+Enter** 逐格查看。这里的合成数据用于说明单元组织和当前机制。

## ① 准备同一组读数

只改变最后一次读数的偏差。参考温度、基础读数与两种方法看到的数据保持一致。参数单位为摄氏度。

In [ ]:
from random import Random
from statistics import mean
from time import perf_counter
from io import BytesIO
from IPython.display import Markdown, Image, display
from matplotlib.figure import Figure

reference = 20.0
n_readings = 9
seed = 7
offset = 8.0  # 之后改成 0，从头运行，观察两种方法的误差
trim_count = 1  # 从最高和最低各去掉多少个读数

In [ ]:
rng = Random(seed)
clean = [round(reference + rng.uniform(-0.4, 0.4), 2) for _ in range(n_readings)]
changed = clean.copy()
changed[-1] += offset
print("原始读数（°C）", clean)
print("含指定偏差的读数（°C）", changed)

## ② 写出并调用本轮方法

普通平均值使用全部读数。下面的方法先排序，去掉两端各 `trim_count` 个读数，再求平均。函数就在本格定义，后面的实验调用它。

先看它对这组具体输入做了什么。读数不足时，两端裁剪会留下空集合，所以函数在这里明确检查该条件。

In [ ]:
def trimmed_mean(values, trim_count):
    if trim_count < 0 or 2 * trim_count >= len(values):
        raise ValueError("裁剪数量需非负，并至少保留一个读数")
    ordered = sorted(values)
    retained = ordered[trim_count:len(ordered) - trim_count]
    return mean(retained)

print("排序后的读数", sorted(changed))
print("保留的读数", sorted(changed)[trim_count:len(changed) - trim_count])
print("普通平均值", round(mean(changed), 3))
print("裁剪平均值", round(trimmed_mean(changed, trim_count), 3))

## ③ 同一输入，怎样计算两种误差？

**绝对误差**就是估计值与参考温度之间相差几度，越小越好。下面把两种方法放在同一个函数里，确保它们使用同一组读数和同一个参考值。

In [ ]:
def compare_estimators(values, reference, trim_count):
    estimates = {"普通平均": mean(values),
                 "裁剪平均": trimmed_mean(values, trim_count)}
    return [{"method": name, "estimate": estimate,
             "absolute_error": abs(estimate - reference)}
            for name, estimate in estimates.items()]

one_comparison = compare_estimators(changed, reference, trim_count)
for result in one_comparison:
    print(result)

## ④ 加入偏差前后，观察是否一致？

用刚才的函数分别处理原始读数和含指定偏差的读数。每次比较都使用相同输入。表格和图只读取这次计算产生的 `rows`。

比较计算时间与绘图、内核启动时间分别记录，方便理解短反馈来自哪一步。

In [ ]:
started = perf_counter()
rows = []
for condition, values in (("原始", clean), ("指定偏差", changed)):
    for result in compare_estimators(values, reference, trim_count):
        rows.append({"condition": condition, **result})
elapsed_s = perf_counter() - started

table = ["| 条件 | 方法 | 估计值（°C） | 绝对误差（°C）↓ |",
         "|:---|:---|---:|---:|"]
for row in rows:
    table.append(f"| {row['condition']} | {row['method']} | "
                 f"{row['estimate']:.3f} | {row['absolute_error']:.3f} |")
display(Markdown("\n".join(table)))
print(f"本次比较计算 {elapsed_s * 1000:.3f} ms（不含绘图和启动）")

In [ ]:
figure = Figure(figsize=(7, 2.8), layout="constrained")
axis = figure.subplots()
labels = ["Original / Mean", "Original / Trimmed", "Changed / Mean", "Changed / Trimmed"]
errors = [row["absolute_error"] for row in rows]
bars = axis.barh(labels, errors, height=0.52, color=["#718598", "#248075"] * 2)
axis.invert_yaxis()
axis.bar_label(bars, fmt="%.3f", padding=5)
axis.set_xlim(0, max(errors + [0.01]) * 1.25)
axis.set_xlabel("Absolute error (°C)")
axis.spines[["top", "right", "left"]].set_visible(False)
axis.tick_params(axis="y", length=0)
buffer = BytesIO()
figure.savefig(buffer, format="png", dpi=130)
display(Image(data=buffer.getvalue()))

## ⑤ 这条观察能支持多少？

下面的文字来自刚才的误差计算。当前数据是一个固定种子的合成小样本，机制和参数范围也很有限。真实测量效果需要独立数据继续检验。

In [ ]:
error_mean = rows[2]["absolute_error"]
error_trimmed = rows[3]["absolute_error"]
if error_trimmed < error_mean:
    observation = "裁剪平均误差较小"
elif error_mean < error_trimmed:
    observation = "普通平均误差较小"
else:
    observation = "两种误差相同"
print(f"当前偏差 {offset:g} °C，这组数据上{observation}。")
print(f"两种绝对误差相差 {abs(error_mean - error_trimmed):.3f} °C。")

## 停在这里，先预测再修改

如果把 `offset` 改成 **0 °C**，你预测两种误差会怎样变化？这会如何影响你对当前方法的判断？

教学使用时先等待回答，再修改参数复跑。之后可根据问题选择新样本或真实测量数据。每一步都沿用当前方法定义与比较函数。